In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import warnings

# Suppress all warnings to keep the output clean
warnings.filterwarnings('ignore')

# Load Dataset 1 (balanced) -This dataset has 21 feature variables and is balanced.
df1 = pd.read_csv('diabetes_binary_5050split_health_indicators_BRFSS2015.csv')
print("--- Dataset 1 (Balanced) ---")
print(df1.head())
print(f"Shape: {df1.shape}")
print(f"Target distribution:\n{df1['Diabetes_binary'].value_counts()}")

# Load Dataset 2 (unbalanced) -This dataset has 21 feature variables and is unbalanced.
df2 = pd.read_csv('diabetes_binary_health_indicators_BRFSS2015.csv')
print("\n--- Dataset 2 (Unbalanced) ---")
print(df2.head())
print(f"Shape: {df2.shape}")
print(f"Target distribution:\n{df2['Diabetes_binary'].value_counts()}")


--- Dataset 1 (Balanced) ---
   Diabetes_binary  HighBP  HighChol  CholCheck   BMI  Smoker  Stroke  \
0              0.0     1.0       0.0        1.0  26.0     0.0     0.0   
1              0.0     1.0       1.0        1.0  26.0     1.0     1.0   
2              0.0     0.0       0.0        1.0  26.0     0.0     0.0   
3              0.0     1.0       1.0        1.0  28.0     1.0     0.0   
4              0.0     0.0       0.0        1.0  29.0     1.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  ...  AnyHealthcare  \
0                   0.0           1.0     0.0  ...            1.0   
1                   0.0           0.0     1.0  ...            1.0   
2                   0.0           1.0     1.0  ...            1.0   
3                   0.0           1.0     1.0  ...            1.0   
4                   0.0           1.0     1.0  ...            1.0   

   NoDocbcCost  GenHlth  MentHlth  PhysHlth  DiffWalk  Sex   Age  Education  \
0          0.0      3.0       5.0     

In [9]:
def run_ml_pipeline(df, is_balanced):
    """
    Runs a complete ML pipeline for a given dataset.
    """
    # Defining features (X) and target (y)
    X = df.drop('Diabetes_binary', axis=1)
    y = df['Diabetes_binary']

    # Split data into training and testing sets
    if is_balanced:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    else:
        # Stratify for unbalanced dataset to maintain class distribution
        # this is to ensure the same ratio of classes in both training and test sets.
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # Initializing and train the Random Forest model
    # Using class weights for the unbalanced dataset
    if not is_balanced:
        model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    else:
        model = RandomForestClassifier(n_estimators=100, random_state=42)

    model.fit(X_train, y_train)

    # Make predictions on the test set
    y_pred = model.predict(X_test)

    # Printing evaluation metrics
    print("\n--- Performance Metrics ---")
    print(classification_report(y_test, y_pred, target_names=['No Diabetes', 'Diabetes']))
    print("\n--- Confusion Matrix ---")
    print(confusion_matrix(y_test, y_pred))

    return model, X_test, y_test


In [8]:
# Train and evaluate model on Dataset 1 (Balanced)
print("\n##############################################")
print("#### Model on Dataset 1 (Balanced 50-50) #####")
print("##############################################")
model1, X_test1, y_test1 = run_ml_pipeline(df1, is_balanced=True)

# Train and evaluate model on Dataset 2 (Unbalanced)
print("\n####################################################")
print("#### Model on Dataset 2 (Unbalanced - Real World) ####")
print("####################################################")
model2, X_test2, y_test2 = run_ml_pipeline(df2, is_balanced=False)



##############################################
#### Model on Dataset 1 (Balanced 50-50) #####
##############################################

--- Performance Metrics ---
              precision    recall  f1-score   support

 No Diabetes       0.76      0.69      0.73      7090
    Diabetes       0.72      0.78      0.75      7049

    accuracy                           0.74     14139
   macro avg       0.74      0.74      0.74     14139
weighted avg       0.74      0.74      0.74     14139


--- Confusion Matrix ---
[[4923 2167]
 [1564 5485]]

####################################################
#### Model on Dataset 2 (Unbalanced - Real World) ####
####################################################

--- Performance Metrics ---
              precision    recall  f1-score   support

 No Diabetes       0.88      0.97      0.92     43667
    Diabetes       0.46      0.16      0.24      7069

    accuracy                           0.86     50736
   macro avg       0.67      0.56      0

1. Create end-to-end ML for both datasets and thoroughly explain the performance of each model. Discuss the performance (recall and precision) metrics in detail.

**The Balanced Model 1** performed consistently well across both classes.
Recall (Diabetes): 78%
Precision (Diabetes): 72%4

I am aware for a medical application, recall is paramount. Model 1's recall of 78% is clinically good in comparison to Model 2. .

**The Ubalanced Model 2** showed a severe performance discrepancy. It correctly identifying the large number of negative case. It failed accurately predict the positive minority class.
Recall (Diabetes): 16%
Precision (Diabetes): 47%

I observed that Model 2's low recall of 16% for the diabetes class is a critical failure. This indicates that using class_weight='balanced' implemented  was not enough to address the extreme class imbalance.

2.Compare the two models and analyze which one you would prefer.

I would not use Model 1 for production. Its strong performance is an illusion created by the artificially balanced training data. Instead I would use a  refined version of Model 2 that uses the realistic dataset. This is a great depediction of real world challenges with data.


In [11]:
# loading again with SMOTE and and RandomForestClassifier
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as imbpipeline

# Load Dataset 2 (unbalanced)
df2 = pd.read_csv('diabetes_binary_health_indicators_BRFSS2015.csv')

# Define features (X) and target (y)
X = df2.drop('Diabetes_binary', axis=1)
y = df2['Diabetes_binary']

# Split data into training and testing sets with stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Create a pipeline with SMOTE and RandomForestClassifier
pipeline_smote = imbpipeline([
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Fitting the pipeline on the training data
pipeline_smote.fit(X_train, y_train)




Pipeline(steps=[('smote', SMOTE(random_state=42)),
                ('classifier', RandomForestClassifier(random_state=42))])

In [12]:
import joblib

# Defining the filename for your saved model
filename = 'best_diabetes_model_smote.joblib'

# Saving the trained pipeline object to the file
joblib.dump(pipeline_smote, filename)

print(f"Model saved successfully to {filename}")


Model saved successfully to best_diabetes_model_smote.joblib


In [13]:
import joblib
import pandas as pd

# Load the saved model object from the file
loaded_model = joblib.load('best_diabetes_model_smote.joblib')

print("Model loaded successfully!")

# Now you can use the loaded model to make predictions on new data
# For example, test data given
new_patient_data = pd.DataFrame([
    {
        'HighBP': 1.0, 'HighChol': 0.0, 'CholCheck': 1.0, 'BMI': 30.0, 'Smoker': 0.0,
        'Stroke': 0.0, 'HeartDiseaseorAttack': 0.0, 'PhysActivity': 1.0, 'Fruits': 1.0,
        'Veggies': 1.0, 'HvyAlcoholConsump': 0.0, 'AnyHealthcare': 1.0, 'NoDocbcCost': 0.0,
        'GenHlth': 3.0, 'MentHlth': 5.0, 'PhysHlth': 10.0, 'DiffWalk': 0.0, 'Sex': 0.0,
        'Age': 8.0, 'Education': 6.0, 'Income': 8.0
    }
])

# Make a prediction using the loaded model
prediction = loaded_model.predict(new_patient_data)

# The prediction will be either 0.0 or 1.0
if prediction[0] == 1.0:
    print("Prediction: Prediabetes or Diabetes")
else:
    print("Prediction: No Diabetes")

# You can also get prediction probabilities
probabilities = loaded_model.predict_proba(new_patient_data)
print(f"Prediction probabilities: {probabilities}")


Model loaded successfully!
Prediction: No Diabetes
Prediction probabilities: [[0.97 0.03]]
